# Support Integrity Auditor (SIA) — Core Pipeline
This notebook demonstrates the complete, end-to-end reproducible pipeline for the Support Integrity Auditor (SIA) system, including data cleaning, self-supervised pseudo-label generation, DeBERTa-v3 classifier training, and deterministic evidence dossier generation.

In [ ]:
# Step 1: Initialize and Clean Raw CRM Data
from src.pipeline_stage1 import IngestionEngine
raw_data_path = 'data/customer_support_tickets.csv'
cleaned_df = IngestionEngine.clean_crm_data(raw_data_path)
print(f'Cleaned dataset shape: {cleaned_df.shape}')
cleaned_df.head()

## Stage 1: Self-Supervised Pseudo-Label Generation
We fuse zero-shot Semantic Urgency ($S_{sem}$) and robust Operational Latency ($S_{ops}$) indicators to derive an inferred severity level, and flag mismatches against the assigned priority column.

In [ ]:
# Step 2: Run Signal Fusion and Generate Pseudo-Labels
from src.pipeline_stage1 import PseudoLabelGenerator
# Process a subset to demonstrate the pipeline
generator = PseudoLabelGenerator(w1=0.6, w2=0.4, threshold=2)
labeled_df = generator.run_pipeline(cleaned_df.head(200))
labeled_df[['Ticket Priority', 'S_sem', 'S_ops', 'inferred_severity_label', 'mismatch_label']].head(10)

## Stage 2: Classifier Inference Demonstration
We load our fine-tuned DeBERTa-v3 model to make mismatch predictions using serialized text representations.

In [ ]:
# Step 3: Run Batch Inference on Flagged Mismatches
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_path = './models/sia_deberta'
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
print('Model loaded successfully.')

## Stage 3: Structured Evidence Dossier Generation
For any identified mismatch, we run our deterministic, hallucination-free generation engine to produce a verified Evidence Dossier matching the target JSON schema.

In [ ]:
# Step 4: Generate Evidence Dossier Sample
from src.dossier_engine import DossierEngine
sample_row = labeled_df.iloc[0].to_dict()
dossier = DossierEngine.generate(sample_row, predicted_mismatch=1, confidence_score=0.942)
import json
print(json.dumps(dossier, indent=2))